# Algorithm Replicator 01: SIRS-to-Sepsis-1 Vertical Slice

## Engineering objective

Reconstruct the current SIRS-to-Sepsis-1 implementation to understand:

1. Why clinical configuration, criterion strategies, temporal adapters,
   episode transformations, and association builders are separate layers.
2. Which abstractions represent genuine variation versus organizational structure.
3. How measurement validity is converted into exact state intervals.
4. Why raw episodes and classification-effective episodes are both retained.
5. Why Sepsis associations are preserved separately from encounter summaries.
6. Where future clinical-definition changes should be implemented.

## Method

For each stage:

1. Identify the design pressure.
2. Implement the transformation directly in Polars.
3. Inspect every intermediate dataframe.
4. State its invariants.
5. Compare it with the production implementation.
6. Identify the effect of a hypothetical clinical-definition change.

In [ ]:
import polars as pl
from datetime import datetime, timedelta
import os
import sys
from pathlib import Path

CWD = Path().cwd()
while 'notebooks' in str(CWD):
    CWD = CWD.parent

sys.path.insert(1, str(CWD))

In [15]:

from src_strategy.configs.aggregator import FEATURE_REGISTRY
from src_strategy.configs.sirscalculator import sirs_config
from src_strategy.configs.shortdurationfilter import (
    sirs_episode_filter_config,
)
from src_strategy.configs.severitysepsis import severitysepsisconfig
from src_strategy.configs.dataconfig import input_filenames_v3, data_config

In [17]:
from src_strategy.configs.dataconfig import bp_config

In [ ]:
from src_strategy.configs.outlierdetection.extremeoutliers import ph

In [16]:
data_config

DataConfig(encounter_col='EncounterEpicCsn', event_dt_col='Event_DateTime', event_name_col='Event_Name', grouper_col='Event_Grouper', type_col='Type', val_col='NumericValue', raw_val_col='Value', BASELINE_CREATININE='Baseline_Creatinine', BASELINE_SBP='Baseline_SBP', BASELINE_RESP_RATE='Baseline_RespiratoryRate', BASELINE_PULSE_RATE='Baseline_PulseRate', BASELINE_PLATELETS='Baseline_Platelets', BASELINE_BILIRUBIN='Baseline_Bilirubin', BASELINE_eGFR='Baseline_eGFR', BASELINE_WBC='Baseline_WBC', unique_subset_with_grouper=['EncounterEpicCsn', 'Event_DateTime', 'Event_Grouper', 'Value'], unique_subset_with_type=['EncounterEpicCsn', 'Event_DateTime', 'Type', 'Value'], unique_subset_with_eventname=['EncounterEpicCsn', 'Event_DateTime', 'Event_Name', 'Value'], unique_subset_with_all=['EncounterEpicCsn', 'Event_DateTime', 'Type', 'Event_Grouper', 'Event_Name', 'Value'], base_cols=['EncounterEpicCsn', 'Event_DateTime', 'Type', 'Event_Grouper', 'Event_Name', 'NumericValue', 'Value'])

In [14]:
from src_strategy.data_ingest.dataloader_1 import DataLoader

In [ ]:
DataLoader(
    input_output_dataconfig=input_filenames_v3,
    data_config=data_config,
    bp_config=bp_config,
)